<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/09-serving-inference/04-monitoring-inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitoring LLM Inference (concept + live server)

**Goal:** Read an inference server's metrics the way an on-call engineer does: parse vLLM's `/metrics`, turn histogram buckets into percentiles and **goodput**, see why "GPU utilization" misleads, and diagnose three incidents from their signals. Then, optionally, load-test a real server on a Colab T4 and watch the metrics move, including a one-line prompt change that breaks the prefix cache.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Why this notebook exists

[Section 08](../08-operations/01-observability-and-llmops.ipynb) traced *one request* through prompts, retrieval, and tools. This is the layer below: the **model server and its GPUs**, where "the first token takes 3 seconds" is either a full KV cache, a longer prompt, a broken prefix cache, or a sick card, and the metrics tell you which.

Everything above the appendix is framework-free Python on canned snapshots and runs on a CPU.

> **⭐ Key takeaway —** monitor three layers (client, server, accelerator) for latency, traffic, saturation, and errors. **Page on goodput** (the share of requests meeting the TTFT and TPOT objectives) and errors; **diagnose** with the server's own signals (queue, KV cache usage, preemptions, prefix-cache hit rate); **autoscale on the queue**, never on GPU utilization.

## What `/metrics` looks like

vLLM serves Prometheus text at `http://<server>:8000/metrics`. Three metric types matter:

- **Counters** only go up (`vllm:generation_tokens_total`). You always look at their **rate** over a window.
- **Gauges** are a current value (`vllm:num_requests_waiting`, `vllm:kv_cache_usage_perc`).
- **Histograms** count observations into cumulative buckets (`_bucket{le="0.5"}` = requests with TTFT ≤ 0.5 s), plus `_sum` and `_count`.

The cell below builds two snapshots, 60 seconds apart, of a healthy replica: an 8B model on an H100 serving a support assistant at 6 requests/s, each with a 3,000-token shared system prompt plus ~500 tokens of conversation and ~300 output tokens. (Metric names as of 2026-11; check your version's `/metrics`.)

In [ ]:
TTFT_LE = [0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 10.0]          # subset of vLLM's TTFT buckets (seconds)
ITL_LE  = [0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5]            # subset of its inter-token-latency buckets

def exposition(c, model="support-8b"):
    """Render cumulative values as Prometheus text, the format a real /metrics returns."""
    L = []
    lab = f'model_name="{model}"'
    for name in ["prompt_tokens_total", "generation_tokens_total", "prefix_cache_queries_total",
                 "prefix_cache_hits_total", "num_preemptions_total"]:
        L.append(f'vllm:{name}{{{lab}}} {c[name]}')
    for reason in ["stop", "length"]:
        L.append(f'vllm:request_success_total{{finished_reason="{reason}",{lab}}} {c["success_" + reason]}')
    for name in ["num_requests_running", "num_requests_waiting", "kv_cache_usage_perc"]:
        L.append(f'vllm:{name}{{{lab}}} {c[name]}')
    for hist, les in [("time_to_first_token_seconds", TTFT_LE), ("inter_token_latency_seconds", ITL_LE)]:
        for le, n in zip(les + ["+Inf"], c[hist]):
            L.append(f'vllm:{hist}_bucket{{le="{le}",{lab}}} {n}')
        L.append(f'vllm:{hist}_count{{{lab}}} {c[hist][-1]}')
    return "\n".join(L)

def snapshot_pair(base, window):
    """t0 = base counters; t1 = base + what happened in a 60 s window; gauges as observed at t1."""
    t0, t1 = dict(base), dict(base)
    for k, v in window.items():
        if isinstance(v, list):
            t1[k] = [a + b for a, b in zip(base[k], v)]
        elif k.endswith("_total") or k.startswith("success_"):
            t1[k] = base[k] + v
        else:
            t1[k] = v
    return exposition(t0), exposition(t1)

BASE = dict(prompt_tokens_total=9_000_000, generation_tokens_total=800_000, prefix_cache_queries_total=9_000_000,
            prefix_cache_hits_total=7_700_000, num_preemptions_total=0, success_stop=2_500, success_length=60,
            num_requests_running=0, num_requests_waiting=0, kv_cache_usage_perc=0.0,
            time_to_first_token_seconds=[1_200, 2_300, 2_520, 2_550, 2_555, 2_560, 2_560, 2_560, 2_560],
            inter_token_latency_seconds=[90_000, 760_000, 795_000, 799_000, 800_000, 800_000, 800_000, 800_000])

HEALTHY = dict(prompt_tokens_total=360 * 3_500, generation_tokens_total=108_000,
               prefix_cache_queries_total=360 * 3_500, prefix_cache_hits_total=360 * 3_000,
               num_preemptions_total=0, success_stop=352, success_length=8,
               num_requests_running=18, num_requests_waiting=0, kv_cache_usage_perc=0.31,
               time_to_first_token_seconds=[150, 330, 360, 360, 360, 360, 360, 360, 360],
               inter_token_latency_seconds=[20_000, 98_000, 107_000, 108_000, 108_000, 108_000, 108_000, 108_000])

t0_text, t1_text = snapshot_pair(BASE, HEALTHY)
print("\n".join(t1_text.splitlines()[:14]), "\n...")

## Parse it (no client library needed)

The text format is one sample per line: `name{label="value",...} number`. A dozen lines of Python turn it into a dictionary keyed by name and labels, which is all the arithmetic below needs.

In [ ]:
import re

LINE = re.compile(r'^([a-zA-Z_:][\w:]*)(?:\{([^}]*)\})?\s+(\S+)$')

def parse(text):
    out = {}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        m = LINE.match(line)
        if not m:
            continue
        name, labels, value = m.groups()
        lab = tuple(sorted(re.findall(r'(\w+)="([^"]*)"', labels or "")))
        out[(name, lab)] = float(value)
    return out

def total(samples, name, **match):
    """Sum every sample of a metric whose labels include `match` (like PromQL's sum())."""
    return sum(v for (n, lab), v in samples.items() if n == name and all((k, str(x)) in lab for k, x in match.items()))

def rate(s0, s1, name, seconds=60, **match):
    return (total(s1, name, **match) - total(s0, name, **match)) / seconds

s0, s1 = parse(t0_text), parse(t1_text)
print(f"requests/s          {(total(s1, 'vllm:request_success_total') - total(s0, 'vllm:request_success_total')) / 60:.1f}")
print(f"prompt tokens/s     {rate(s0, s1, 'vllm:prompt_tokens_total'):,.0f}")
print(f"generation tokens/s {rate(s0, s1, 'vllm:generation_tokens_total'):,.0f}")
print(f"prefix hit rate     {rate(s0, s1, 'vllm:prefix_cache_hits_total') / rate(s0, s1, 'vllm:prefix_cache_queries_total'):.0%}")
print(f"preemptions/s       {rate(s0, s1, 'vllm:num_preemptions_total'):.2f}")
print(f"running / waiting   {total(s1, 'vllm:num_requests_running'):.0f} / {total(s1, 'vllm:num_requests_waiting'):.0f}")
print(f"KV cache usage      {total(s1, 'vllm:kv_cache_usage_perc'):.0%}")

## Percentiles and goodput from buckets

A histogram stores *cumulative* counts, so percentiles are **estimates**: Prometheus's `histogram_quantile` finds the bucket holding the target rank and interpolates linearly inside it. Goodput against an objective that sits **on a bucket boundary** is exact. Here's both, written out.

In [ ]:
def buckets(s0, s1, hist):
    """Per-window cumulative counts by upper bound, summed across label sets (PromQL: sum by (le) (rate(...)))."""
    by_le = {}
    for (n, lab), v in s1.items():
        if n == hist + "_bucket":
            le = dict(lab)["le"]
            by_le[le] = by_le.get(le, 0) + v - s0.get((n, lab), 0)
    return sorted(((float(le), c) for le, c in by_le.items()), key=lambda x: x[0])

def histogram_quantile(q, bs):
    total_n = bs[-1][1]
    rank = q * total_n
    prev_le, prev_c = 0.0, 0.0
    for le, c in bs:
        if c >= rank:
            if le == float("inf"):
                return prev_le                       # Prometheus returns the highest finite bound
            return prev_le + (le - prev_le) * (rank - prev_c) / (c - prev_c)
        prev_le, prev_c = le, c

def goodput(bs, objective):
    for le, c in bs:
        if le == objective:
            return c / bs[-1][1]
    raise ValueError(f"{objective} s is not a bucket boundary: goodput would be interpolated")

TTFT_SLO, TPOT_SLO = 1.0, 0.05

ttft = buckets(s0, s1, "vllm:time_to_first_token_seconds")
itl = buckets(s0, s1, "vllm:inter_token_latency_seconds")
print(f"TTFT p50 {histogram_quantile(0.5, ttft):.2f} s   p95 {histogram_quantile(0.95, ttft):.2f} s   "
      f"goodput (<= {TTFT_SLO} s) {goodput(ttft, TTFT_SLO):.1%}")
print(f"TPOT p50 {histogram_quantile(0.5, itl) * 1000:.0f} ms  p95 {histogram_quantile(0.95, itl) * 1000:.0f} ms  "
      f"goodput (<= {TPOT_SLO * 1000:.0f} ms) {goodput(itl, TPOT_SLO):.1%}")

# The worked example from the topic: 1,000 requests, p95 falls in the 1.0-2.5 s bucket.
example = [(0.25, 600), (0.5, 820), (0.75, 900), (1.0, 940), (2.5, 995), (float("inf"), 1000)]
print(f"\nworked example: p95 = {histogram_quantile(0.95, example):.2f} s (anywhere in 1.0-2.5 s), "
      f"goodput <= 1.0 s = {goodput(example, 1.0):.0%} (exact)")

> **🚩 Common mistake —** averaging per-replica percentiles, or graphing one "latency". Sum the **buckets** across replicas first (`sum by (le)`), then take the quantile; and keep TTFT and TPOT on separate lines, because a fix for one often hurts the other.

## The "GPU utilization" trap

`nvidia-smi`'s utilization (DCGM's `DCGM_FI_DEV_GPU_UTIL`) is the share of time **any kernel was running**. A decoding server always has a kernel running, so it reads ~100% whether the card is at 5% or 95% of what it can do. Memory used is no better: vLLM reserves ~90% of memory for its KV cache at start-up. The useful question is how much of the **bandwidth** decode is using (MBU), and you can estimate it from server metrics alone.

In [ ]:
WEIGHTS_GB, KV_MB_PER_TOKEN, SPEC_BW_TBS = 16.1, 0.131, 3.35     # 8B BF16 model on an H100

def mbu(gen_tok_s, running, avg_ctx):
    steps_per_s = gen_tok_s / running                             # each step yields one token per running request
    bytes_per_step = (WEIGHTS_GB + running * avg_ctx * KV_MB_PER_TOKEN / 1e3) * 1e9
    return steps_per_s * bytes_per_step / (SPEC_BW_TBS * 1e12)

print(f"one user at 120 tok/s:                 MBU {mbu(120, 1, 500):.0%}   (nvidia-smi says ~100%)")
print(f"32 requests, 2,000 ctx, 3,000 tok/s:  MBU {mbu(3000, 32, 2000):.0%}")

## Diagnose three incidents

Each scenario below is a 60-second window you might see during a page. The rules in `diagnose()` are the five patterns from the topic, written as code. Run it, then check each verdict against the raw signals yourself before trusting it.

In [ ]:
def signals(window):
    a, b = snapshot_pair(BASE, window)
    s0, s1 = parse(a), parse(b)
    t, i = buckets(s0, s1, "vllm:time_to_first_token_seconds"), buckets(s0, s1, "vllm:inter_token_latency_seconds")
    q = rate(s0, s1, "vllm:prefix_cache_queries_total")
    return dict(ttft_p95=histogram_quantile(0.95, t), tpot_p95=histogram_quantile(0.95, i),
                ttft_goodput=goodput(t, TTFT_SLO), tpot_goodput=goodput(i, TPOT_SLO),
                waiting=total(s1, "vllm:num_requests_waiting"), running=total(s1, "vllm:num_requests_running"),
                kv=total(s1, "vllm:kv_cache_usage_perc"), preempt=rate(s0, s1, "vllm:num_preemptions_total"),
                hit=rate(s0, s1, "vllm:prefix_cache_hits_total") / q if q else 0.0,
                prompt_per_req=rate(s0, s1, "vllm:prompt_tokens_total") / rate(s0, s1, "vllm:request_success_total"))

def diagnose(s, normal_hit=0.85, normal_prompt_per_req=3_500):
    found = []
    if s["ttft_goodput"] < 0.95 and s["waiting"] > 0 and s["kv"] > 0.9:
        found.append("out of capacity: queue + full KV cache -> scale out, re-check sizing inputs")
    if s["hit"] < normal_hit / 2:
        found.append("prefix cache collapse -> look for a per-request change at the top of the prompt")
    if s["prompt_per_req"] > 1.5 * normal_prompt_per_req:
        found.append("longer prompts -> find the retrieval/history change, cap context")
    if s["tpot_goodput"] < 0.95 and s["ttft_goodput"] >= 0.95:
        found.append("batch too large for the TPOT objective -> lower --max-num-seqs or scale out")
    if s["preempt"] > 0:
        found.append("KV cache overflowing (preemptions) -> lower batch/context, FP8 KV cache")
    return found or ["healthy"]

SCENARIOS = {
    "healthy": HEALTHY,
    "A: Monday 9 a.m.": dict(HEALTHY, prompt_tokens_total=600 * 3_500, prefix_cache_queries_total=600 * 3_500,
        prefix_cache_hits_total=600 * 3_000, success_stop=590, success_length=10, num_preemptions_total=40, num_requests_running=150,
        num_requests_waiting=38, kv_cache_usage_perc=0.98,
        time_to_first_token_seconds=[40, 110, 200, 260, 300, 420, 540, 590, 600],
        inter_token_latency_seconds=[5_000, 60_000, 150_000, 170_000, 178_000, 180_000, 180_000, 180_000]),
    "B: after Tuesday's release": dict(HEALTHY, prefix_cache_hits_total=360 * 60,
        num_requests_waiting=15, kv_cache_usage_perc=0.45,
        time_to_first_token_seconds=[10, 40, 90, 140, 200, 352, 360, 360, 360]),
    "C: new max-num-seqs": dict(HEALTHY, num_requests_running=120, kv_cache_usage_perc=0.86,
        inter_token_latency_seconds=[1_000, 20_000, 70_000, 95_000, 104_000, 108_000, 108_000, 108_000]),
}

for name, w in SCENARIOS.items():
    s = signals(w)
    print(f"{name}\n  TTFT p95 {s['ttft_p95']:.2f}s (good {s['ttft_goodput']:.0%})  TPOT p95 {s['tpot_p95'] * 1000:.0f}ms "
          f"(good {s['tpot_goodput']:.0%})  waiting {s['waiting']:.0f}  running {s['running']:.0f}  KV {s['kv']:.0%}  "
          f"preempt/s {s['preempt']:.2f}  hit {s['hit']:.0%}")
    for d in diagnose(s):
        print("   ->", d)

Scenario B is the costly one in real life: request rate, errors, and TPOT are all unchanged, so a dashboard without the hit-rate panel points at "add GPUs". The release put a timestamp on the first line of the system prompt; every prompt became unique, so none of its 3,000 shared tokens were served from cache, and prefill work rose ~7×.

> **⚠️ Production reality —** put anything that varies per request (timestamps, user names, request IDs, retrieved documents) **after** the stable part of the prompt. A prefix cache reuses only the longest identical prefix.

## Alerts as rules

Page on what users feel, ticket the causes. Written as data, the rules are easy to review and to evaluate against a snapshot:

In [ ]:
ALERTS = [   # (name, severity, PromQL you would deploy, check against our signals)
    ("TTFT goodput below objective", "page",
     'sum(rate(vllm:time_to_first_token_seconds_bucket{le="1.0"}[5m])) / sum(rate(vllm:time_to_first_token_seconds_count[5m])) < 0.95',
     lambda s: s["ttft_goodput"] < 0.95),
    ("TPOT goodput below objective", "page",
     'sum(rate(vllm:inter_token_latency_seconds_bucket{le="0.05"}[5m])) / sum(rate(vllm:inter_token_latency_seconds_count[5m])) < 0.95',
     lambda s: s["tpot_goodput"] < 0.95),
    ("Sustained saturation", "ticket",
     'sum(vllm:num_requests_waiting) > 0 and max(vllm:kv_cache_usage_perc) > 0.9',
     lambda s: s["waiting"] > 0 and s["kv"] > 0.9),
    ("Preemptions", "ticket", 'sum(rate(vllm:num_preemptions_total[15m])) > 0', lambda s: s["preempt"] > 0),
    ("Prefix cache collapse", "ticket",
     'rate(vllm:prefix_cache_hits_total[30m]) / rate(vllm:prefix_cache_queries_total[30m]) < 0.5 * (same, offset 1w)',
     lambda s: s["hit"] < 0.85 / 2),
]

for name, w in SCENARIOS.items():
    s = signals(w)
    fired = [f"{sev}: {a}" for a, sev, _, check in ALERTS if check(s)]
    print(f"{name:<28} {fired or 'no alerts'}")

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Separate TTFT and TPOT panels and objectives | One "latency" line that hides which phase regressed |
| Page on goodput and errors | Page on GPU utilization or queue length thresholds |
| Set objectives on bucket boundaries | An objective between buckets, so goodput is interpolated |
| Sum buckets across replicas, then take the quantile | Average each pod's p95 |
| Watch the prefix-cache hit rate | Discover a prompt change by buying GPUs |
| Autoscale on waiting requests per replica | Autoscale on CPU or GPU utilization |
| Alert on Xid, ECC, and clock throttling | Let one sick card drag a replica for days |

## Optional appendix — watch a live server (needs a GPU)

> **⚠️ This appendix needs a GPU runtime and is skippable.** In Colab: **Runtime → Change runtime type → T4 GPU**. On CPU, read it and move on; the analysis above is the graded material.

It stands up its own vLLM server (same launch as notebook 02), then runs load at two concurrency levels with a long **shared** system prompt, scraping `/metrics` before and after each run with the parser above. The last run adds a timestamp to the top of the system prompt, and you watch the prefix-cache hit rate and TTFT respond.

In [ ]:
# Appendix cell 1 — GPU check. Stop here if this prints "no GPU".
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — the analysis above is what's graded; skip this appendix")

In [ ]:
# Appendix cell 2 — install vLLM + fix the torchaudio CUDA-version clash (see notebook 01 for why).
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"
# If the line above errors, Runtime -> Restart session, then rerun from cell 1 SKIPPING this %pip line.

In [ ]:
# Appendix cell 3 — launch the server with prefix caching on.
import subprocess, time, urllib.request

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"          # the one place the model name lives
LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--max-model-len", "2048", "--port", "8000", "--enforce-eager", "--gpu-memory-utilization", "0.85",
     "--enable-prefix-caching"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(90):
    if server.poll() is not None:
        print("server exited; log tail:"); print(open(LOG).read()[-3000:]); break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2); print("server is up"); break
    except Exception:
        time.sleep(5)

In [ ]:
# Appendix cell 4 — scrape helper and a load generator that samples the gauges while it runs.
import threading, concurrent.futures, datetime
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

def scrape():
    return parse(urllib.request.urlopen("http://localhost:8000/metrics", timeout=5).read().decode())

names = sorted({n for n, _ in scrape() if n.startswith("vllm:")})
needed = ["vllm:time_to_first_token_seconds_bucket", "vllm:inter_token_latency_seconds_bucket",
          "vllm:prefix_cache_hits_total", "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc"]
missing = [n for n in needed if n not in names]
print("all metrics present" if not missing else f"missing in this vLLM version: {missing}\navailable: {names}")

POLICY = ("Refunds are available within 30 days of purchase with a receipt. Store credit is offered after 30 days. "
          "Opened software, gift cards, and final-sale items cannot be returned. Escalate safety issues immediately. ")
SYSTEM = "You are a support assistant for a hardware store. Policy: " + POLICY * 30     # ~1,200 shared tokens

def one_request(system):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": "Can I return an opened drill after 40 days?"}]
    for chunk in client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=64, stream=True):
        pass

def run(concurrency, n_requests, timestamped=False):
    peak = {"waiting": 0, "kv": 0}
    stop = threading.Event()
    def sample():
        while not stop.is_set():
            s = scrape()
            peak["waiting"] = max(peak["waiting"], total(s, "vllm:num_requests_waiting"))
            peak["kv"] = max(peak["kv"], total(s, "vllm:kv_cache_usage_perc"))
            time.sleep(0.5)
    before = scrape(); t0 = time.perf_counter()
    sampler = threading.Thread(target=sample); sampler.start()
    def system():   # the regression: a per-request timestamp on the FIRST line
        return (f"Current time: {datetime.datetime.now().isoformat()}\n" + SYSTEM) if timestamped else SYSTEM
    with concurrent.futures.ThreadPoolExecutor(concurrency) as ex:
        list(ex.map(lambda _: one_request(system()), range(n_requests)))
    secs = time.perf_counter() - t0; stop.set(); sampler.join(); after = scrape()
    t = buckets(before, after, "vllm:time_to_first_token_seconds")
    i = buckets(before, after, "vllm:inter_token_latency_seconds")
    q = rate(before, after, "vllm:prefix_cache_queries_total", secs)
    ms = lambda bs, p: histogram_quantile(p, bs) * 1000 if bs and bs[-1][1] else float("nan")   # nan if metric absent
    print(f"conc {concurrency:>2} {'timestamped' if timestamped else 'stable     '}  "
          f"TTFT p50 {ms(t, 0.5):>5.0f} ms p95 {ms(t, 0.95):>5.0f} ms  "
          f"TPOT p95 {ms(i, 0.95):>4.0f} ms  "
          f"gen {rate(before, after, 'vllm:generation_tokens_total', secs):>5.0f} tok/s  "
          f"hit {rate(before, after, 'vllm:prefix_cache_hits_total', secs) / q if q else 0:>4.0%}  "
          f"peak waiting {peak['waiting']:.0f}  peak KV {peak['kv']:.0%}")

run(1, 4)                                   # warm-up: fills the prefix cache

In [ ]:
# Appendix cell 5 — the experiment. Watch which signals move between runs.
run(1, 20)
run(16, 96)
run(16, 96, timestamped=True)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

What to look for:

- **Concurrency 1 → 16:** generation tokens/s rises several times (batching), TPOT rises a little, TTFT rises (requests now share steps and may wait), and peak waiting may leave zero.
- **Timestamped prompt:** the hit rate collapses toward 0% and TTFT rises, while TPOT barely moves. That's scenario B, on a real server. On this small model the prefill is cheap, so the effect is milliseconds; on an 8B or 70B model with real traffic it's seconds.
- **`nvidia-smi` utilization** reads high in every run, which is why it told you nothing.

In [ ]:
# Appendix cell 6 — free the GPU (a connected runtime still uses Colab GPU quota: disconnect when done).
server.terminate()
time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

## Exercises

1. **A fourth incident.** Write a window for "one replica of four has a throttled GPU" (what happens to its tokens/s, TPOT, and running requests compared with the others?) and add a rule to `diagnose()` that catches it without false-firing on the healthy scenario.
2. **Objective off a boundary.** Change `TTFT_SLO` to 0.8 s and explain the error you get. What are your two options, and which would you choose for production?
3. **Aggregate correctly.** Create two replicas' bucket lists where averaging their p95s gives a different answer from summing buckets first. How far apart can they get?
4. **Autoscaling rule.** Using `signals()`, write `desired_replicas(current, s)` that adds a replica when waiting per replica exceeds 3 and removes one when waiting is 0 and KV usage is under 30%. Which scenario does it scale, and is that the right call for scenario B?
5. **Live MBU.** In the appendix, use the measured generation tokens/s and running requests with `mbu()` (change the constants to TinyLlama's 2.2 GB of weights and the T4's 0.32 TB/s). How far from the bandwidth ceiling is the small model, and why?